# GOaT selection notebook

Run top to bottom on Colab. Every stage writes its artifacts to Google Drive
and skips itself when those artifacts already exist.

## Step 1. Open the Colab notebook

## Step 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Step 3. Git clone the repo, shallow

In [ ]:
![ -d /content/GOaT/.git ] || git clone --depth 1 https://github.com/champyod/GOAT.git /content/GOaT

## Step 4. Install dependencies

In [ ]:
%pip install -q transformers datasets sentencepiece peft accelerate jiwer sacrebleu opencv-python-headless scipy psutil tqdm matplotlib paddleocr paddlepaddle

## Step 5. Import the package from the clone

In [ ]:
import sys

sys.path.insert(0, "/content/GOaT/model/src")

from goat_model import constants as c

## Step 6. Stage datasets, Drive checked, MT and OCR selection

In [ ]:
from pathlib import Path

from goat_model.data import (
    download_flores200,
    download_scbmt,
    download_thaiocr_evaluation,
    download_thaiocrbench,
)

DATA_ROOT = Path("/content/drive/MyDrive/GOaT/datasets")
MT_DIR = DATA_ROOT / "mt"
OCR_DIR = DATA_ROOT / "ocr"

if not (MT_DIR / "train.th").is_file():
    download_scbmt(MT_DIR)

if not (MT_DIR / "test" / "flores200.th").is_file():
    download_flores200(MT_DIR / "test")

if not (OCR_DIR / "thai-ocr-evaluation" / "images").is_dir():
    download_thaiocr_evaluation(OCR_DIR / "thai-ocr-evaluation")

if not (OCR_DIR / "thaiocrbench" / "images").is_dir():
    download_thaiocrbench(OCR_DIR / "thaiocrbench")

## Step 7. MT model selection

Methodology, model selection MT. Both NLLB candidates zero shot EN->TH on
FLORES-200, batch 16, 5 repeats. Rule, NLLB-600M iff BLEU > 35 and latency
<= 2 s. Result lands in results/mt_selection.json on Drive.

In [ ]:
from pathlib import Path

from goat_model.constants import (
    LANG_CODES,
    MT_BATCH_SIZE,
    MT_BEAM_SIZE,
    MT_BLEU_THRESHOLD,
    MT_LATENCY_THRESHOLD_S,
    MT_MAX_LENGTH,
    MT_MODELS,
    MT_N_RUNS,
    SEED,
)
from goat_model.metrics import paired_t_test, summarize
from goat_model.mt.engine import get_mt
from goat_model.mt.evaluate import load_pairs, run_mt
from goat_model.utils import setup_seed, write_json

setup_seed(SEED)

MT_SELECTION = Path("/content/drive/MyDrive/GOaT/results/mt_selection.json")

if MT_SELECTION.is_file():
    print(f"skipped - already selected: {MT_SELECTION}")
else:
    src, refs, _ = load_pairs(MT_DIR / "test" / "flores200.en", MT_DIR / "test" / "flores200.th")

    bleu_runs, lat_runs = {}, {}
    for model in MT_MODELS:
        backend = get_mt(model, LANG_CODES["en"], LANG_CODES["th"],
                         beam=MT_BEAM_SIZE, max_length=MT_MAX_LENGTH)
        bleu_runs[model], lat_runs[model] = [], []
        for _ in range(MT_N_RUNS):
            r = run_mt(backend, src, refs, batch_size=MT_BATCH_SIZE)
            bleu_runs[model].append(r["bleu"])
            lat_runs[model].append(r["average_ms_per_sentence"] / 1000)

    m600 = summarize(bleu_runs["NLLB-200-distilled-600M"])[0]
    l600 = summarize(lat_runs["NLLB-200-distilled-600M"])[0]
    selected = ("NLLB-200-distilled-600M"
                if m600 > MT_BLEU_THRESHOLD and l600 <= MT_LATENCY_THRESHOLD_S
                else "NLLB-200-distilled-1.3B")

    write_json(
        MT_SELECTION,
        {
            "models": {
                m: {"bleu": summarize(bleu_runs[m]), "avg_s_per_sent": summarize(lat_runs[m])}
                for m in MT_MODELS
            },
            "paired_t_test": paired_t_test(bleu_runs[MT_MODELS[0]], bleu_runs[MT_MODELS[1]]),
            "decision_rule": f"{MT_MODELS[0]} iff BLEU > {MT_BLEU_THRESHOLD} "
                             f"and latency <= {MT_LATENCY_THRESHOLD_S}s",
            "selected": selected,
        },
    )

## Step 8. OCR model selection

Methodology, model selection OCR. Both candidates on both public benchmark
sets, CER and latency, 5 repeats. Rule, ThaiTrOCR iff mean CER <= 0.10,
else lowest CER. Result lands in results/ocr_selection.json on Drive.

In [ ]:
from pathlib import Path

from goat_model.constants import (
    OCR_ALPHA,
    OCR_CER_THRESHOLD,
    OCR_DATASETS,
    OCR_IMG_SIZE,
    OCR_MODELS,
    OCR_N_RUNS,
    SEED,
)
from goat_model.metrics import cohens_d, paired_t_test
from goat_model.ocr.engine import get_ocr
from goat_model.ocr.evaluate import aggregate_records, discover_assets, run_ocr
from goat_model.utils import setup_seed, write_json

setup_seed(SEED)

OCR_SELECTION = Path("/content/drive/MyDrive/GOaT/results/ocr_selection.json")

if OCR_SELECTION.is_file():
    print(f"skipped - already selected: {OCR_SELECTION}")
else:
    per_dataset, cer_flat = {}, {m: [] for m in OCR_MODELS}
    for model in OCR_MODELS:
        per_dataset[model] = {}
        backend = get_ocr(model)
        for dataset in OCR_DATASETS:
            assets = discover_assets(OCR_DIR / dataset)
            runs = [run_ocr(backend, assets, OCR_IMG_SIZE[model]) for _ in range(OCR_N_RUNS)]
            per_dataset[model][dataset] = aggregate_records(runs)
            cer_flat[model] += [rec["cer"] for run in runs for rec in run]

    thai_mean = sum(cer_flat["ThaiTrOCR"]) / len(cer_flat["ThaiTrOCR"])
    pp_mean = sum(cer_flat["PP-OCRv5-mobile"]) / len(cer_flat["PP-OCRv5-mobile"])
    selected_ocr = ("ThaiTrOCR" if thai_mean <= OCR_CER_THRESHOLD
                    else ("ThaiTrOCR" if thai_mean < pp_mean else "PP-OCRv5-mobile"))

    write_json(
        OCR_SELECTION,
        {
            "n_runs": OCR_N_RUNS,
            "models": per_dataset,
            "comparisons": [{
                "a": "ThaiTrOCR", "b": "PP-OCRv5-mobile",
                "paired_t_test": paired_t_test(
                    cer_flat["ThaiTrOCR"], cer_flat["PP-OCRv5-mobile"], alpha=OCR_ALPHA
                ),
                "cohens_d": cohens_d(cer_flat["ThaiTrOCR"], cer_flat["PP-OCRv5-mobile"]),
            }],
            "decision_rule": f"ThaiTrOCR iff mean CER <= {OCR_CER_THRESHOLD}, else lowest CER",
            "mean_cer_thaitrocr": thai_mean,
            "mean_cer_ppocrv5": pp_mean,
            "selected": selected_ocr,
        },
    )

## Step 9. Fine-tune selected MT model

Methodology, fine-tuning. Only NLLB-200-distilled-600M is fine-tuned; the
NLLB-1.3B candidate stays zero shot. Freeze all original weights, LoRA on
q_proj and v_proj, grid rank x alpha x learning rate, epochs 3 to 10, Adam,
FLORES-200 BLEU measured every epoch, pick the config with the highest BLEU.
Result lands in results/mt_training.json on Drive.

In [ ]:
# Step 9. LoRA fine-tune the selected MT model (post-selection, EN->TH)
# Methodology: freeze all weights, LoRA on q_proj/v_proj, grid r x alpha x LR,
# epochs 3-10, Adam, BLEU on FLORES-200 every epoch, pick highest BLEU.
# Only NLLB-600M is fine-tuned; NLLB-1.3B stays zero-shot.
import json
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from goat_model.constants import (
    LANG_CODES,
    LORA_ALPHAS,
    LORA_EPOCHS,
    LORA_LEARNING_RATES,
    LORA_RANKS,
    LORA_TARGET_MODULES,
    MT_BATCH_SIZE,
    MT_MAX_LENGTH,
    SEED,
)
from goat_model.metrics import corpus_bleu
from goat_model.mt.engine import NLLB_HF_IDS
from goat_model.mt.evaluate import load_pairs
from goat_model.utils import setup_seed, write_json

setup_seed(SEED)

MT_SELECTION = Path("/content/drive/MyDrive/GOaT/results/mt_selection.json")
MT_TRAINING = Path("/content/drive/MyDrive/GOaT/results/mt_training.json")
MT_DIR = Path("/content/drive/MyDrive/GOaT/datasets/mt")
MT_OUT_ROOT = Path("/content/drive/MyDrive/GOaT/artifacts/mt_lora")

MODEL_600M = "NLLB-200-distilled-600M"

if MT_TRAINING.is_file():
    print(f"skipped - already trained: {MT_TRAINING}")
else:
    selected = json.loads(MT_SELECTION.read_text())["selected"]
    if selected != MODEL_600M:
        print(f"no fine-tune needed - selected {selected} stays zero-shot")
    else:
        model_id = NLLB_HF_IDS[MODEL_600M]
        tokenizer = AutoTokenizer.from_pretrained(
            model_id, src_lang=LANG_CODES["en"], tgt_lang=LANG_CODES["th"]
        )

        def load_split(name):
            src, ref, _ = load_pairs(MT_DIR / f"{name}.en", MT_DIR / f"{name}.th")
            model_inputs = tokenizer(
                src, max_length=MT_MAX_LENGTH, truncation=True, padding=False
            )
            labels = tokenizer(
                text_target=ref, max_length=MT_MAX_LENGTH, truncation=True, padding=False
            )
            labels_ids = [
                [tok if tok != tokenizer.pad_token_id else -100 for tok in ids]
                for ids in labels["input_ids"]
            ]
            return Dataset.from_dict(
                {
                    "input_ids": model_inputs["input_ids"],
                    "attention_mask": model_inputs["attention_mask"],
                    "labels": labels_ids,
                }
            )

        train_ds = load_split("train")
        val_ds = load_split("val")

        flores_src, flores_ref, _ = load_pairs(
            MT_DIR / "test" / "flores200.en", MT_DIR / "test" / "flores200.th"
        )

        def compute_bleu(eval_preds):
            preds, labels = eval_preds
            if isinstance(preds, tuple):
                preds = preds[0]
            preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
            decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
            return {"bleu": round(corpus_bleu(decoded_labels, decoded_preds), 4)}

        grid_results = {}
        best = None
        for r in LORA_RANKS:
            for alpha in LORA_ALPHAS:
                for lr in LORA_LEARNING_RATES:
                    setup_seed(SEED)
                    base = AutoModelForSeq2SeqLM.from_pretrained(model_id)
                    base = base.to("cuda")
                    peft_config = LoraConfig(
                        task_type=TaskType.SEQ_2_SEQ_LM,
                        r=r,
                        lora_alpha=alpha,
                        target_modules=list(LORA_TARGET_MODULES),
                        lora_dropout=0.1,
                        bias="none",
                    )
                    model = get_peft_model(base, peft_config)
                    model.print_trainable_parameters()

                    collator = DataCollatorForSeq2Seq(tokenizer, model=model)

                    out_dir = MT_OUT_ROOT / f"r{r}_alpha{alpha}_lr{lr}"
                    args = Seq2SeqTrainingArguments(
                        output_dir=str(out_dir),
                        learning_rate=lr,
                        per_device_train_batch_size=MT_BATCH_SIZE,
                        num_train_epochs=LORA_EPOCHS[1],
                        optim="adamw_torch",
                        eval_strategy="epoch",
                        save_strategy="epoch",
                        save_total_limit=1,
                        load_best_model_at_end=True,
                        metric_for_best_model="eval_bleu",
                        greater_is_better=True,
                        predict_with_generate=True,
                        seed=SEED,
                    )
                    trainer = Seq2SeqTrainer(
                        model=model,
                        args=args,
                        train_dataset=train_ds,
                        eval_dataset=val_ds,
                        tokenizer=tokenizer,
                        data_collator=collator,
                        compute_metrics=compute_bleu,
                    )
                    trainer.train(resume_from_checkpoint=True)
                    model.save_pretrained(out_dir)

                    ft_model = PeftModel.from_pretrained(
                        AutoModelForSeq2SeqLM.from_pretrained(model_id).to("cuda"),
                        out_dir,
                    )
                    ft_model.eval()
                    hypotheses = []
                    with torch.inference_mode():
                        for i in range(0, len(flores_src), MT_BATCH_SIZE):
                            batch = tokenizer(
                                flores_src[i : i + MT_BATCH_SIZE],
                                return_tensors="pt",
                                padding=True,
                                truncation=True,
                            ).to("cuda")
                            gen = ft_model.generate(
                                **batch,
                                forced_bos_token_id=tokenizer.convert_tokens_to_ids(
                                    LANG_CODES["th"]
                                ),
                                num_beams=4,
                                max_length=MT_MAX_LENGTH,
                            )
                            hypotheses.extend(
                                tokenizer.batch_decode(gen, skip_special_tokens=True)
                            )
                    flores_bleu = corpus_bleu(flores_ref, hypotheses)
                    del ft_model, base, model
                    torch.cuda.empty_cache()

                    key = {"rank": r, "alpha": alpha, "lr": lr}
                    grid_results[f"r{r}_alpha{alpha}_lr{lr}"] = {
                        **key,
                        "flores_bleu": flores_bleu,
                        "adapter": str(out_dir),
                    }
                    print(f"config r{r} alpha{alpha} lr{lr}: FLORES BLEU {flores_bleu}")
                    if best is None or flores_bleu > best[0]:
                        best = (flores_bleu, key)

        write_json(
            MT_TRAINING,
            {
                "selected": selected,
                "base_model": model_id,
                "target_modules": list(LORA_TARGET_MODULES),
                "epochs": list(LORA_EPOCHS),
                "grid_results": grid_results,
                "winner": best[1],
                "winner_flores_bleu": best[0],
            },
        )
        print(f"wrote {MT_TRAINING}")